# MedQwen3B-Reasoner: Medical Reasoning 

**MedQwen3B-Reasoner** - 一个专门针对医学领域推理和数学问题求解优化的3B参数语言模型。本指南将指导您完成模型的微调和部署过程，该模型结合了临床专业知识和结构化推理能力。
**Key Tutorial Focuses**:
- 利用 GRPO (Group Relative Policy Optimization) 进行医学领域适配
-  精选的训练数据集合，包含 PubMedQA (70%) 和数学推理数据集
- 使用 `<reasoning>`/`<answer>` 格式实现结构化推理输出
- 通过 unsloth 实现 4-bit 量化的高效部署
- 在临床决策支持和生物医学研究分析中的实际应用

In [2]:
%%capture
!pip install unsloth vllm
!pip install --upgrade pillow
# If you are running this notebook on local, you need to install `diffusers` too
!pip install diffusers
# Temporarily install a specific TRL nightly version
!pip install ipywidgets
!pip install diffusers

我们将在本教程中使用优秀的 Unsloth 库。

In [1]:
from unsloth import FastLanguageModel, PatchFastRL
PatchFastRL("GRPO", FastLanguageModel)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 02-22 16:12:03 __init__.py:190] Automatically detected platform cuda.


## 下载并初始化模型
我们首先将下载模型，并利用 50% 的 GPU 容量以及 vLLM 推理来加速使用 Qlora 的 GRPO 训练。


In [7]:
from unsloth import is_bfloat16_supported
import torch
max_seq_length = 2048 # Can increase for longer reasoning traces
lora_rank = 64 # Larger rank = smarter, but slower

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/home/aeon/.cache/huggingface/hub/models--Qwen--Qwen2.5-3B-Instruct/snapshots/aa8e72537993ba99e69dfaafa59ed015b17504d1",
    max_seq_length = max_seq_length,
    load_in_4bit = True, # False for LoRA 16bit
    fast_inference = True, # Enable vLLM fast inference
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.5, # Reduce if out of memory
    
)



model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ], # Remove QKVO if out of memory
    lora_alpha = lora_rank,
    use_gradient_checkpointing = "unsloth", # Enable long context finetuning
    random_state = 3407,
)
# 配置 PEFT (Parameter-Efficient Fine-Tuning) 模型：
#使用 LoRA 进行参数高效微调
#指定需要微调的模块：
#Transformer 的注意力机制相关模块 (q_proj, k_proj, v_proj, o_proj)
#MLP 相关模块 (gate_proj, up_proj, down_proj)

==((====))==  Unsloth 2025.2.12: Fast Qwen2 patching. Transformers: 4.48.3.
   \\   /|    GPU: NVIDIA RTX A6000. Max memory: 47.536 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 8.6. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.28.post3. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading /home/aeon/.cache/huggingface/hub/models--Qwen--Qwen2.5-3B-Instruct/snapshots/aa8e72537993ba99e69dfaafa59ed015b17504d1 with actual GPU utilization = 47.37%
Unsloth: Your GPU has CUDA compute capability 8.6 with VRAM = 47.54 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 2048. Num Sequences = 288.
Unsloth: vLLM's KV Cache can use up to 16.55 GB. Also swap space = 5 GB.
INFO 02-22 16:22:07 config.py:542] This model supports multiple tasks: {'generate', 'embed', 'classify', 'score', 'reward'}.

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


INFO 02-22 16:22:12 model_runner.py:1115] Loading model weights took 5.7622 GB
INFO 02-22 16:22:12 punica_selector.py:18] Using PunicaWrapperGPU.
INFO 02-22 16:22:22 worker.py:267] Memory profiling takes 9.68 seconds
INFO 02-22 16:22:22 worker.py:267] the current vLLM instance can use total_gpu_memory (47.54GiB) x gpu_memory_utilization (0.47) = 22.52GiB
INFO 02-22 16:22:22 worker.py:267] model weights take 5.76GiB; non_torch_memory takes 0.02GiB; PyTorch activation peak memory takes 1.57GiB; the rest of the memory reserved for KV Cache is 15.17GiB.
INFO 02-22 16:22:22 executor_base.py:110] # CUDA blocks: 27611, # CPU blocks: 9102
INFO 02-22 16:22:22 executor_base.py:115] Maximum concurrency for 2048 tokens per request: 215.71x
INFO 02-22 16:22:29 model_runner.py:1434] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI. If out-of-memory error

Capturing CUDA graph shapes: 100%|██████████| 39/39 [00:37<00:00,  1.03it/s]

INFO 02-22 16:23:07 model_runner.py:1562] Graph capturing finished in 38 secs, took 2.68 GiB
INFO 02-22 16:23:07 llm_engine.py:431] init engine (profile, create kv cache, warmup model) took 55.05 seconds



Unsloth 2025.2.12 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


## 持续预训练

现在我们进行持续微调。我们将分别使用来自 huggingface hub 的三个数据集：`openai/gsm8k`、`qiaojin/PubMedQA` 和 `esilhealth/Health_Benchmarks`。如代码所示，我们在 PubMedQA 的情况下过滤了上下文的长度，因为它可能有更长的文本，这可能会导致我们的训练内存不足（在本教程中，我们的目标是使用具有 16/24 GB 内存的 T4 或 A10 GPU）。

另请注意，过滤后我们从 `PubmedQA` 数据集中获得了几乎三倍的样本。这是有意为之的，因为这对模型来说是一个更具挑战性的数据集，因此，我们希望模型能更频繁地看到这些数据。

In [9]:
import re
from datasets import load_dataset, Dataset, interleave_datasets, concatenate_datasets

# 系统提示
SYSTEM_PROMPT = """
Respond in the following format:
<reasoning>
...
</reasoning>
<answer>
...
</answer>
"""

# XML格式模板
XML_COT_FORMAT = """\
<reasoning>
{reasoning}
</reasoning>
<answer>
{answer}
</answer>
"""

# 辅助函数：提取XML格式中的答案
def extract_xml_answer(text: str) -> str:
    answer = text.split("<answer>")[-1]
    answer = answer.split("</answer>")[0]
    return answer.strip()

# 辅助函数：提取带有####标记的答案
def extract_hash_answer(text: str) -> str | None:
    if "####" not in text:
        return None
    return text.split("####")[1].strip()

# 数据处理函数uncomment middle messages for 1-shot prompting
def get_datasets(split = "train") -> Dataset:
    data = load_dataset('openai/gsm8k', 'main')[split] # type: ignore
    data = data.map(lambda x: { # type: ignore
        'prompt': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': x['question']}
        ],
        'answer': extract_hash_answer(x['answer']),
        'db_set':'gsm8k'
    }) # type: ignore
    data = data.remove_columns(['question'])

    data_qa = load_dataset("qiaojin/PubMedQA", "pqa_artificial")[split] # two times more than other datasets
    data_qa = data_qa.filter(lambda x: len("\n".join(x['context']['contexts'])) < 1024) # avoid long traces
    data_qa = data_qa.map(lambda x: { # type: ignore
        'prompt': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {
                "role": "user",
                "content": "Given the scientific context below:\n" +
                          "\n".join(x['context']['contexts']) +
                          "\n\nAnswer the following question:\n" +
                          x['question'] +
                          " with 'yes', 'no' or 'maybe'. You need to carefully review the context and reason before answering."
            },
        ],
        'answer': x['final_decision'],
        'db_set': 'pubmedqa'
    }) # type: ignore
    data_qa = data_qa.remove_columns(['pubid', 'question', 'context', 'long_answer', 'final_decision'])


    categories =['Lab_Medicine', 'Wearables', 'Dermatology', 'Gastroenterology', 'Internal_Medicine', 'Oncology', 'Orthopedics', 'General_Surgery', 'Ophthalmology', 'Audiology', 'Head_Neck_Surgery', 'Elderly_Care', 'Pediatrics', 'Allergy_Immunology', 'Rheumatology', 'Pharmacy', 'Obstetrics_Gynecology', 'Microbiology', 'Dentistry', 'Physical_Medicine_and_Rehabilitation', 'Neurology', 'Psychiatry', 'Pathology', 'Genetics', 'Rare_Diseases', 'Hematology', 'Emergency', 'Endocrinology', 'Radiology', 'Cardiology', 'Pulmonology', 'Infectious_Diseases', 'Critical_Care', 'Pediatric_Surgery', 'Neuroscience', 'Epidemiology', 'Fitness_Sports', 'Health_Education', 'Health_Economics', 'Health_Entrepreneurship', 'Hospital_Management', 'Mental_Health', 'Nutrition', 'Palliative_Care', 'Preventive_Medicine', 'Public_Health', 'Social_Media_Addiction', 'Sleep', 'Supplements', 'Vaccination', 'Work_Health', 'Wellbeing']
    data_mc = concatenate_datasets([load_dataset("yesilhealth/Health_Benchmarks",i)[i] for i in categories])
    data_mc = data_mc.map(lambda x: { # type: ignore
        'prompt': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {
                "role": "user",
                "content": "\n\nAnswer the following question:\n" +
                          x['Questions'] +
                          "\n With 'A', 'B', 'C' or 'D'. You need to carefully review the context and reason before answering."
            },
        ],
        'answer': x['Answers'],
        'db_set': 'med_mc'
    }) # type: ignore
    data_mc = data_mc.remove_columns(['Answers', 'Questions'])

    dataset = concatenate_datasets([data, data_qa, data_mc])
    return dataset


In [10]:
dataset = get_datasets()
dataset = dataset.shuffle(seed=42)
train_test_split = dataset.train_test_split(test_size=0.1)
#将10%的数据用作测试集
train_dataset = train_test_split["train"]
test_dataset = train_test_split["test"]
print(f"train size: {len(train_dataset)}, test size: {len(test_dataset)}")

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Map:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/211269 [00:00<?, ? examples/s]

Filter:   0%|          | 0/211269 [00:00<?, ? examples/s]

...

Generating Vaccination split:   0%|          | 0/130 [00:00<?, ? examples/s]

Generating Work_Health split:   0%|          | 0/130 [00:00<?, ? examples/s]

Generating Wellbeing split:   0%|          | 0/110 [00:00<?, ? examples/s]

Map:   0%|          | 0/7535 [00:00<?, ? examples/s]

train size: 38171, test size: 4242


# 设计奖励函数

我个人认为，使用 GRPO 获得良好性能的诀窍是设计良好的奖励函数。就像我们教狗做一些技巧时一样，我们希望在模型完成困难任务时给予更高的奖励，在完成较小任务时给予较小的奖励。这意味着我们将尝试教导模型关于我们想要它响应的格式（如 `reasoning`）以及其响应的质量和正确性。

让我们快速回顾以下几点：

## correctness_reward_func

这个函数确保最终答案是正确的。在 `gsm8k` 的情况下，有时模型会回答 `The final answer is $80.` 在这种情况下它不会完全匹配真实答案 `80`，因此 `a in r` 检查在某种程度上捕获了这种情况，但奖励只有 1 分，因为我们不想鼓励冗长。对于其他数据集，我们简单地接受答案，因为在 `pubmedqa` 的情况下，答案是 `yes`、`no` 或 `maybe`，而在 `health_benchmarks` 的情况下是多选题。

其他奖励函数确保格式的正确性，使模型以适当的 `reasoning` 和 `answer` 标签响应。



In [11]:
## Reward functions
def correctness_reward_func(prompts, completions, answer, db_set, **kwargs) -> list[float]:
    responses = [completion[0]['content'] for completion in completions]
    q = prompts[0][-1]['content']
    extracted_responses = [extract_xml_answer(r) for r in responses]
    print('-'*20, f"Question:\n{q}", f"\nAnswer:\n{answer[0]}", f"\nResponse:\n{responses[0]}", f"\nExtracted:\n{extracted_responses[0]}")
    rewards = []
    for r,a,dt in zip(extracted_responses, answer, db_set):
        if dt == "gsm8k":
            if a in r:
                rewards.append(1.0)
            elif r == a:
                rewards.append(2.0)
            else:
                rewards.append(0.0)
        else:
            rewards.append(2.0 if r.lower() == a.strip().lower() else 0.0)
    return rewards


def int_reward_func(completions, db_set, **kwargs) -> list[float]:
    responses = [completion[0]['content'] for completion in completions]
    extracted_responses = [extract_xml_answer(r) for r in responses]
    rewards = []
    for r,dt in zip(extracted_responses,db_set):
        if dt == "gsm8k":
            rewards.append(0.5 if r.isdigit() else 0.0)
        elif dt == "pubmedqa":
            rewards.append(0.5 if ('yes' in r.lower() or 'no' in r.lower() or 'maybe' in r.lower()) else 0.0)
        else:
            rewards.append(0.5 if ('a' in r.lower() or 'b' in r.lower() or 'c' in r.lower() or 'd' in r.lower()) else 0.0)
    return rewards

def strict_format_reward_func(completions, **kwargs) -> list[float]:
    """Reward function that checks if the completion has a specific format."""
    pattern = r"^<reasoning>\n.*?\n</reasoning>\n<answer>\n.*?\n</answer>\n$"
    responses = [completion[0]["content"] for completion in completions]
    matches = [re.match(pattern, r) for r in responses]
    return [0.5 if match else 0.0 for match in matches]

def soft_format_reward_func(completions, **kwargs) -> list[float]:
    """Reward function that checks if the completion has a specific format."""
    pattern = r"<reasoning>.*?</reasoning>\s*<answer>.*?</answer>"
    responses = [completion[0]["content"] for completion in completions]
    matches = [re.match(pattern, r) for r in responses]
    return [0.5 if match else 0.0 for match in matches]

def count_xml(text) -> float:
    count = 0.0
    if text.count("<reasoning>\n") == 1:
        count += 0.125
    if text.count("\n</reasoning>\n") == 1:
        count += 0.125
    if text.count("\n<answer>\n") == 1:
        count += 0.125
        count -= len(text.split("\n</answer>\n")[-1])*0.001
    if text.count("\n</answer>") == 1:
        count += 0.125
        count -= (len(text.split("\n</answer>")[-1]) - 1)*0.001
    return count

def xmlcount_reward_func(completions, **kwargs) -> list[float]:
    contents = [completion[0]["content"] for completion in completions]
    return [count_xml(c) for c in contents]

# 设置训练参数

我们将使用来自 huggingface 的 TRL 库，它支持 GRPO。


In [12]:
from trl import GRPOConfig, GRPOTrainer
training_args = GRPOConfig(
    use_vllm = True, # 使用vLLM加速推理for fast inference!
    # 优化器参数
    learning_rate = 5e-6,
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    weight_decay = 0.1,
    warmup_ratio = 0.1,
    lr_scheduler_type = "cosine",# 学习率调度器类型
    optim = "adamw_8bit",# 优化器类型使用8位精度的AdamW优化器
    # 训练参数
    logging_steps = 1,
    bf16 = is_bfloat16_supported(),
    fp16 = not is_bfloat16_supported(),
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 1, # Increase to 4 for smoother training
    num_generations = 6, # Decrease if out of memory
    # 序列长度限制
    max_prompt_length = 1024,
    max_completion_length = 1024,
    #num_train_epochs = 1, # Set to 1 for a full training run
    max_steps = 750,
    save_steps = 100,
    max_grad_norm = 0.1,
    report_to = "none", # # 不使用外部报告工具wanddb
    output_dir = "outputs", # 输出目录 
)

Unsloth: We know expect `per_device_train_batch_size` to be a multiple of `num_generations`.
We will change the batch size of 1 to the `num_generations` of 6


In [13]:
trainer = GRPOTrainer(
    model = model,# 要训练的模型
    processing_class = tokenizer,# 分词器
    reward_funcs = [# 奖励函数列表
        xmlcount_reward_func,# 检查XML标签数量和位置
        soft_format_reward_func,# 检查XML标签格式   
        strict_format_reward_func,# 检查XML标签格式
        int_reward_func,# 检查答案是否为整数
        correctness_reward_func,# 检查答案是否正确
    ],#最大可能奖励：约4.0
    args = training_args,# 之前定义的训练参数
    train_dataset = train_dataset,# 训练数据集
    eval_dataset=test_dataset,# 评估数据集
)
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs = 1
   \\   /|    Num examples = 38,171 | Num Epochs = 1
O^O/ \_/ \    Batch size per device = 6 | Gradient Accumulation steps = 1
\        /    Total batch size = 6 | Total steps = 750
 "-____-"     Number of trainable parameters = 119,734,272


-------------------- Question:
Given the scientific context below:
Preventing or reducing amyloid-beta (Aβ) accumulation in the brain is an important therapeutic strategy for Alzheimer's disease (AD). Recent studies showed that the water channel aquaporin-4 (AQP4) mediates soluble Aβ clearance from the brain parenchyma along the paravascular pathway. However the direct evidence for roles of AQP4 in the pathophysiology of AD remains absent.
Here, we reported that the deletion of AQP4 exacerbated cognitive deficits of 12-moth old APP/PS1 mice, with increases in Aβ accumulation, cerebral amyloid angiopathy and loss of synaptic protein and brain-derived neurotrophic factor in the hippocampus and cortex. Furthermore, AQP4 deficiency increased atrophy of astrocytes with significant decreases in interleukin-1 beta and nonsignificant decreases in interleukin-6 and tumor necrosis factor-alpha in hippocampal and cerebral samples.

Answer the following question:
Does deletion of aquaporin-4 in AP

Step,Training Loss,reward,reward_std,completion_length,kl,rewards / xmlcount_reward_func,rewards / soft_format_reward_func,rewards / strict_format_reward_func,rewards / int_reward_func,rewards / correctness_reward_func
1,0.000000,2.112020,0.540352,191.666672,0.000000,-0.388672,0.000000,0.000000,0.500000,2.000000
2,-0.000000,0.163086,0.544147,267.000000,0.000000,-0.503906,0.000000,0.000000,0.000000,0.667969
3,0.000000,0.597514,0.555734,172.333344,0.000677,-0.069336,0.000000,0.000000,0.000000,0.667969
4,0.000000,1.115723,1.287359,191.666672,0.000710,-0.384766,0.000000,0.000000,0.500000,1.000000
5,0.000000,0.587240,0.462906,397.666687,0.000222,-0.412109,0.000000,0.000000,0.000000,1.000000
6,0.000000,0.606771,0.117387,240.333344,0.000553,-0.392578,0.000000,0.000000,0.000000,1.000000
7,0.000000,1.212565,1.270948,217.833344,0.000786,-0.621094,0.000000,0.000000,0.500000,1.335938
8,0.000000,0.786784,0.162931,171.666672,0.000463,-0.212891,0.000000,0.000000,0.000000,1.000000
9,0.000000,1.600586,0.327006,254.000000,0.000850,-0.898438,0.000000,0.000000,0.500000,2.000000
10,0.000000,0.624674,0.155120,261.500000,0.000282,-0.375000,0.000000,0.000000,0.000000,1.000000


-------------------- Question:
Kyle has a newspaper-delivery route. Every Monday through Saturday, he delivers the daily paper for the 100 houses on his route. On Sunday, 10 of his customers do not get the Sunday paper, but he delivers 30 papers to other houses that get the newspaper only on Sunday. How many papers does Kyle deliver each week? 
Answer:
720 
Response:
<reasoning>
Kyle delivers newspapers to 100 houses every weekday (Monday through Saturday). So, over these six days, Kyle delivers a total of 100 newspapers/day * 6 days = 600 newspapers. On Sunday, 10 houses do not receive the Sunday paper, which means 100 - 10 = 90 houses get the newspaper that Sunday. Additionally, an extra 30 houses receive the Sunday paper that he does not deliver newspapers to the regular 6 days, making the total Sunday delivery 90 + 30 = 120 papers. Thus, adding the Sunday deliveries to the regular deliveries gives 600 + 120 = 720 newspapers delivered in a week.

</reasoning>
<answer>
Kyle delivers 

TrainOutput(global_step=750, training_loss=0.01433052039474488, metrics={'train_runtime': 3098.4405, 'train_samples_per_second': 1.452, 'train_steps_per_second': 0.242, 'total_flos': 0.0, 'train_loss': 0.01433052039474488})

# 测试时间

首先我们将在没有 `Qlora` heads 的情况下测试我们的模型。然后我们将添加 head 并进行比较。


In [14]:
# 1. 准备输入
text = tokenizer.apply_chat_template([
    {"role" : "user", "content" : "Is Aspirin good for cardio vascular function?"},
], tokenize = False, add_generation_prompt = True)

from vllm import SamplingParams
# 2. 设置采样参数
sampling_params = SamplingParams(
    temperature = 0.8,
    top_p = 0.95,
    max_tokens = 1024,
)
# 3. 生成回答
output = model.fast_generate(
    [text],
    sampling_params = sampling_params,
    lora_request = None,
)[0].outputs[0].text

output

Processed prompts: 100%|██████████| 1/1 [00:04<00:00,  4.76s/it, est. speed input: 8.19 toks/s, output: 77.32 toks/s]


"Aspirin can have cardiovascular benefits, but it's important to consider its potential risks and proper use. Here's a summary:\n\n**Cardiovascular Benefits:**\n- Aspirin can reduce the risk of blood clots, which can help prevent heart attacks and strokes in people with certain cardiovascular conditions.\n- It can also reduce the risk of thrombosis (blood clot formation) in the veins and arteries.\n\n**Risks and Considerations:**\n- **Risk of Bleeding:** Aspirin can increase the risk of bleeding, including gastrointestinal bleeding and bleeding in the brain. This risk is higher in people with existing bleeding disorders or taking other medications that can increase bleeding risk.\n- **Side Effects:** Aspirin can cause side effects such as stomach pain, nausea, and allergic reactions. It can also interact with other medications.\n- **Individuals at Risk:** Aspirin use is not recommended for everyone. People with a history of bleeding disorders, stomach ulcers, or asthma should avoid or 

## 让我们添加 Qlora 权重

添加我们刚刚微调的 Qlora 权重来看看差异


In [15]:
model.save_lora("grpo_saved_lora")

In [16]:
# 1. 准备输入（添加了系统提示）
text = tokenizer.apply_chat_template([
    {"role" : "system", "content" : SYSTEM_PROMPT},
    {"role" : "user", "content" : "Is Aspirin good for cardio vascular function?"},
], tokenize = False, add_generation_prompt = True)

from vllm import SamplingParams
# 2. 设置采样参数（与之前相同）
sampling_params = SamplingParams(
    temperature = 0.8,
    top_p = 0.95,
    max_tokens = 1024,
)
# 3. 生成回答（使用LoRA权重）
output = model.fast_generate(
    text,
    sampling_params = sampling_params,
    lora_request = model.load_lora("grpo_saved_lora"),
)[0].outputs[0].text

output

Processed prompts: 100%|██████████| 1/1 [00:03<00:00,  3.87s/it, est. speed input: 11.65 toks/s, output: 66.52 toks/s]


'<reasoning>\nAspirin is a non-steroidal anti-inflammatory drug (NSAID) that is commonly used for its analgesic, antipyretic, and anti-inflammatory effects. It has also been shown to have some antithrombotic properties, which means it can help prevent blood clots. However, the use of aspirin for cardiovascular health is not without risks. Low-dose aspirin (75-325 mg) is sometimes prescribed for people who have had a heart attack, stroke, or transient ischemic attack (TIA), or those who have other cardiovascular risk factors, to reduce the risk of future cardiovascular events. Nonetheless, aspirin use can lead to side effects such as bleeding, including gastrointestinal bleeding and bleeding in the brain (stroke).\n</reasoning>\n<answer>\nThe use of aspirin for cardio vascular function is generally recommended at low doses (75-325 mg) for individuals who have had a heart attack, stroke, or transient ischemic attack (TIA), or those with other cardiovascular risk factors. However, aspirin

In [17]:
model.save_pretrained_merged("model", tokenizer)

Unsloth: Merging 4bit and LoRA weights to 16bit...
Unsloth: Will use up to 19.99 out of 62.54 RAM for saving.
Unsloth: Saving model... This might take 5 minutes ...


100%|██████████| 36/36 [00:00<00:00, 47.50it/s]


Unsloth: Saving tokenizer... Done.
Done.


# 推送到 huggingface hub

如果你想把你微调的模型推送到 hub，只需要：

In [18]:
model.push_to_hub_merged("myMedModel", tokenizer, token = "hf_*******************")

Unsloth: Merging 4bit and LoRA weights to 16bit...
Unsloth: Will use up to 19.99 out of 62.54 RAM for saving.
Unsloth: Saving model... This might take 5 minutes ...


100%|██████████| 36/36 [00:00<00:00, 97.03it/s]


Unsloth: Saving to organization with address AEONA/myMedModel
Unsloth: Saving tokenizer... Done.
Unsloth: Saving to organization with address AEONA/myMedModel
Unsloth: Uploading all files... Please wait...


RuntimeError: Error while uploading 'model-00001-of-00002.safetensors' to the Hub.